# Google Maps Review Scraper

Notebook untuk **scraping** review dari Google Maps menggunakan Apify API.

**Output:** `data/raw_review/additional/{nama_desa}_reviews_raw.csv` (data mentah)


## 1. Setup & Import

In [1]:
# Jalankan sekali untuk install
# !pip install apify-client

In [2]:
import os
import pandas as pd
from apify_client import ApifyClient

BASE_DIR = os.path.dirname(os.getcwd())
OUTPUT_DIR = os.path.join(BASE_DIR, "data", "raw_review", "additional", "raw")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Output directory: {OUTPUT_DIR}")
print("Setup OK!")

Output directory: d:\Kuliah\TA\TA_Notebook\data\raw_review\additional\raw
Setup OK!


## 2. Konfigurasi

In [ ]:
# ============================================================
# KONFIGURASI — Edit bagian ini
# ============================================================
API_TOKEN = ""   # API token dari https://apify.com → Settings → Integrations
MAX_REVIEWS = 300     # Max review yang diambil per desa wisata

# Daftar desa wisata yang mau di-scrape.
# Isi "url" dengan URL Google Maps masing-masing desa.
# File yang sudah ada di OUTPUT_DIR akan di-skip otomatis.
PLACES = [

    # ── Desa Baru — isi URL Google Maps ──────────────────────
    # SUMATERA
    {"url": "https://maps.app.goo.gl/8JfNy3j4QEWZ95Aj8",  "nama": "Gampong Nusa",              "provinsi": "Aceh"},
    {"url": "https://maps.app.goo.gl/MQyLWComWBJhaUk39",  "nama": "Tomok",                     "provinsi": "Sumatera Utara"},
    {"url": "https://maps.app.goo.gl/zriStkBxYxXsX5MY9",  "nama": "Nagari Pariangan",           "provinsi": "Sumatera Barat"},
    {"url": "https://maps.app.goo.gl/jthLUpXRYhVJP3SE9",  "nama": "Buluh Cina",                 "provinsi": "Riau"},
    {"url": "https://maps.app.goo.gl/QV3vUQL1UgtsBXCs8",  "nama": "Kampung Terih",              "provinsi": "Kepulauan Riau"},
    {"url": "https://maps.app.goo.gl/Paeu2udwsw6CCcyV9",  "nama": "Tanjung Laut",               "provinsi": "Jambi"},
    {"url": "https://maps.app.goo.gl/hHna1cjtRRXNX9WY7",  "nama": "Pulau Kumayan Kota",         "provinsi": "Bengkulu"},
    {"url": "https://maps.app.goo.gl/GP2WgAY9Q6KzMKfs5",  "nama": "Pulau Kemaro",               "provinsi": "Sumatera Selatan"},
    {"url": "https://maps.app.goo.gl/btjJKyZswcWTiLMXA",  "nama": "Mangrove Kurau",             "provinsi": "Kepulauan Bangka Belitung"},
    {"url": "https://maps.app.goo.gl/WKqxpXaLC8rW4zgq5",  "nama": "Kunjir",                     "provinsi": "Lampung"},

    # JAWA
    {"url": "https://maps.app.goo.gl/C9bM1kqsYrabqpMBA",  "nama": "Setu Babakan",               "provinsi": "DKI Jakarta"},
    {"url": "https://maps.app.goo.gl/sYz6hvtZT2DKB6vM6",  "nama": "Kampung Naga",               "provinsi": "Jawa Barat"},
    {"url": "https://maps.app.goo.gl/S6EaS8vDKuP92yMU9",  "nama": "Kampung Marengo Baduy",      "provinsi": "Banten"},

    # NUSA TENGGARA
    {"url": "https://maps.app.goo.gl/CQoZRFBt6XPEfLJo9",  "nama": "Desa Sade",                  "provinsi": "Nusa Tenggara Barat"},
    {"url": "https://maps.app.goo.gl/cK2fW1TFYz2TraR76",  "nama": "Desa Wisata Waerebo",        "provinsi": "Nusa Tenggara Timur"},

    # KALIMANTAN
    {"url": "https://maps.app.goo.gl/Z6kDazDDKwp6zbBB6",  "nama": "Desa Wisata Sungai Utik",    "provinsi": "Kalimantan Barat"},
    {"url": "https://maps.app.goo.gl/Wh112zdaepK6toH49",  "nama": "Desa Wisata Sungai Sekonyer","provinsi": "Kalimantan Tengah"},
    {"url": "https://maps.app.goo.gl/syiFWsazeWyrBRWj7",  "nama": "Kampung Ketupat",            "provinsi": "Kalimantan Selatan"},
    {"url": "https://maps.app.goo.gl/WacSM8q8HnKtLXDq7",  "nama": "Desa Wisata Pampang",        "provinsi": "Kalimantan Timur"},
    {"url": "https://maps.app.goo.gl/1awJoYiGdPo7dbmYA",  "nama": "Desa Wisata Setulang",       "provinsi": "Kalimantan Utara"},

    # SULAWESI
    {"url": "https://maps.app.goo.gl/8XYSFs1wDoH1X3Xq6",  "nama": "Desa Wisata Budo",           "provinsi": "Sulawesi Utara"},
    {"url": "https://maps.app.goo.gl/AyPSzDL48ARPktts5",  "nama": "Desa Wisata Olele",          "provinsi": "Gorontalo"},
    {"url": "https://maps.app.goo.gl/zYQYjiPzCWAK59UY8",  "nama": "Danau Paisu",                "provinsi": "Sulawesi Tengah"},
    {"url": "https://maps.app.goo.gl/kdseHaXPQ5zxsGuKA",  "nama": "Mamuju City",                "provinsi": "Sulawesi Barat"},
    {"url": "https://maps.app.goo.gl/qgnoDCefC914PKKY6",  "nama": "Danau Napabale",             "provinsi": "Sulawesi Tenggara"},

    # MALUKU & PAPUA
    {"url": "https://maps.app.goo.gl/SG8spK2yBmopSfmV7",  "nama": "Pantai Batu Kuda",           "provinsi": "Maluku"},
    {"url": "https://maps.app.goo.gl/HhynjdN6dmc9mM839",  "nama": "Tanjung Rappa Pelangi",      "provinsi": "Maluku Utara"},
    {"url": "https://maps.app.goo.gl/Q1DFutFHR6TiXtEM7",  "nama": "Sauwandarek",                "provinsi": "Papua Barat"},
    {"url": "https://maps.app.goo.gl/aVtadxAvMiK8qDZ4A",  "nama": "Danau Emtofe",               "provinsi": "Papua"},
]

client = ApifyClient(API_TOKEN)
print(f"Total desa wisata dalam list : {len(PLACES)}")
print(f"Max reviews per desa         : {MAX_REVIEWS}")
print(f"Output dir                   : {OUTPUT_DIR}")


Total desa wisata dalam list : 29
Max reviews per desa         : 300
Output dir                   : d:\Kuliah\TA\TA_Notebook\data\raw_review\additional\raw


## 3. Batch Scraping

In [4]:
def scrape_place(client, url, nama, provinsi, max_reviews):
    """Scrape reviews for one place via Apify. Returns DataFrame."""
    run_input = {
        "startUrls": [{"url": url}],
        "maxReviews": max_reviews,
        "reviewsSort": "newest",
        "language": "id",
        "scrapeReviewsPersonalData": True,
        "maxImages": 0,
        "maxQuestions": 0,
        "scrapeContacts": False,
        "scrapeSocialMediaProfiles": {
            "facebooks": False, "instagrams": False,
            "youtubes": False, "tiktoks": False, "twitters": False,
        },
        "scrapeDirectories": False,
        "includeWebResults": False,
        "scrapeTableReservationProvider": False,
    }
    run = client.actor("nwua9Gu5YrADL7ZDj").call(run_input=run_input)

    raw = []
    for item in client.dataset(run["defaultDatasetId"]).iterate_items():
        reviews_list = item.get("reviews", [])
        if not reviews_list:
            text = item.get("reviewText", "") or item.get("text", "") or ""
            if text.strip():
                raw.append({
                    "review_text": text.strip(),
                    "rating":      item.get("stars", item.get("reviewRating", 0)),
                    "author":      item.get("reviewerName", item.get("name", "")),
                    "date":        item.get("publishedAtDate", item.get("reviewDate", "")),
                })
        else:
            for r in reviews_list:
                text = r.get("text", "") or r.get("reviewText", "") or ""
                if text.strip():
                    raw.append({
                        "review_text": text.strip(),
                        "rating":      r.get("stars", r.get("rating", 0)),
                        "author":      r.get("name", r.get("reviewerName", "")),
                        "date":        r.get("publishedAtDate", r.get("reviewDate", "")),
                    })

    df = pd.DataFrame(raw)
    if not df.empty:
        df["nama desa wisata"] = nama
        df["provinsi"] = provinsi
    return df


# ── Batch Scraping Loop ──────────────────────────────────────
results_summary = []

for i, place in enumerate(PLACES, 1):
    nama     = place["nama"]
    url      = place["url"]
    provinsi = place.get("provinsi", "")
    filename = nama.lower().replace(" ", "_") + "_reviews_raw.csv"
    out_path = os.path.join(OUTPUT_DIR, filename)

    print(f"\n[{i}/{len(PLACES)}] {nama} ({provinsi})")

    if not url:
        print(f"  ⚠ URL kosong — skip")
        results_summary.append({"nama": nama, "provinsi": provinsi, "status": "skip (no url)", "count": 0})
        continue

    if os.path.exists(out_path):
        existing = pd.read_csv(out_path)
        print(f"  ✓ Sudah ada ({len(existing)} reviews) — skip")
        results_summary.append({"nama": nama, "provinsi": provinsi, "status": "skip (exists)", "count": len(existing)})
        continue

    try:
        print(f"  → Scraping dari Apify... (tunggu beberapa menit)")
        df = scrape_place(client, url, nama, provinsi, MAX_REVIEWS)
        if df.empty:
            print(f"  ✗ 0 reviews ditemukan")
            results_summary.append({"nama": nama, "provinsi": provinsi, "status": "error (0 reviews)", "count": 0})
        else:
            df.to_csv(out_path, index=False, encoding="utf-8")
            print(f"  ✓ Saved {len(df)} reviews → {filename}")
            results_summary.append({"nama": nama, "provinsi": provinsi, "status": "ok", "count": len(df)})
    except Exception as e:
        print(f"  ✗ Error: {e}")
        results_summary.append({"nama": nama, "provinsi": provinsi, "status": f"error: {str(e)[:80]}", "count": 0})

print("\n" + "="*60)
print("BATCH SELESAI")
print("="*60)



[1/29] Gampong Nusa (Aceh)
  → Scraping dari Apify... (tunggu beberapa menit)


[apify.crawler-google-places runId:PpxjWesf1p1z0AhTZ] -> Status: RUNNING, Message: 
[apify.crawler-google-places runId:PpxjWesf1p1z0AhTZ] -> 2026-05-06T13:46:00.860Z ACTOR: Pulling container image of build t94zr25jkhfIVPY08 from registry.
[apify.crawler-google-places runId:PpxjWesf1p1z0AhTZ] -> 2026-05-06T13:46:00.863Z ACTOR: Creating container.
[apify.crawler-google-places runId:PpxjWesf1p1z0AhTZ] -> 2026-05-06T13:46:00.919Z ACTOR: Starting container.
[apify.crawler-google-places runId:PpxjWesf1p1z0AhTZ] -> 2026-05-06T13:46:00.920Z ACTOR: Running under "LIMITED_PERMISSIONS" permission level.
[apify.crawler-google-places runId:PpxjWesf1p1z0AhTZ] -> 2026-05-06T13:46:01.875Z INFO  System info {"apifyVersion":"3.5.3","apifyClientVersion":"2.19.0","crawleeVersion":"3.16.0","osType":"Linux","nodeVersion":"v24.15.0"}
[apify.crawler-google-places runId:PpxjWesf1p1z0AhTZ] -> 2026-05-06T13:46:02.296Z INFO  [BG ENQUEUE] Finished enqueueing
[apify.crawler-google-places runId:PpxjWesf1p1z0AhTZ] ->

  ✓ Saved 61 reviews → gampong_nusa_reviews_raw.csv

[2/29] Tomok (Sumatera Utara)
  → Scraping dari Apify... (tunggu beberapa menit)


[apify.crawler-google-places runId:XYPqL1Ip80dM5WJre] -> Status: RUNNING, Message: 
[apify.crawler-google-places runId:XYPqL1Ip80dM5WJre] -> 2026-05-06T13:50:39.639Z ACTOR: Pulling container image of build t94zr25jkhfIVPY08 from registry.
[apify.crawler-google-places runId:XYPqL1Ip80dM5WJre] -> 2026-05-06T13:50:39.642Z ACTOR: Creating container.
[apify.crawler-google-places runId:XYPqL1Ip80dM5WJre] -> 2026-05-06T13:50:39.877Z ACTOR: Starting container.
[apify.crawler-google-places runId:XYPqL1Ip80dM5WJre] -> 2026-05-06T13:50:39.878Z ACTOR: Running under "LIMITED_PERMISSIONS" permission level.
[apify.crawler-google-places runId:XYPqL1Ip80dM5WJre] -> 2026-05-06T13:50:40.894Z INFO  System info {"apifyVersion":"3.5.3","apifyClientVersion":"2.19.0","crawleeVersion":"3.16.0","osType":"Linux","nodeVersion":"v24.15.0"}
[apify.crawler-google-places runId:XYPqL1Ip80dM5WJre] -> 2026-05-06T13:50:41.593Z INFO  [BG ENQUEUE] Finished enqueueing
[apify.crawler-google-places runId:XYPqL1Ip80dM5WJre] ->

  ✓ Saved 31 reviews → tomok_reviews_raw.csv

[3/29] Nagari Pariangan (Sumatera Barat)
  → Scraping dari Apify... (tunggu beberapa menit)


...

  ✓ Saved 122 reviews → tanjung_rappa_pelangi_reviews_raw.csv

[28/29] Sauwandarek (Papua Barat)
  → Scraping dari Apify... (tunggu beberapa menit)


[apify.crawler-google-places runId:spwHOiXRDgimae07c] -> Status: RUNNING, Message: 
[apify.crawler-google-places runId:spwHOiXRDgimae07c] -> 2026-05-06T14:19:38.728Z ACTOR: Pulling container image of build t94zr25jkhfIVPY08 from registry.
[apify.crawler-google-places runId:spwHOiXRDgimae07c] -> 2026-05-06T14:19:38.730Z ACTOR: Creating container.
[apify.crawler-google-places runId:spwHOiXRDgimae07c] -> 2026-05-06T14:19:38.799Z ACTOR: Starting container.
[apify.crawler-google-places runId:spwHOiXRDgimae07c] -> 2026-05-06T14:19:38.799Z ACTOR: Running under "LIMITED_PERMISSIONS" permission level.
[apify.crawler-google-places runId:spwHOiXRDgimae07c] -> 2026-05-06T14:19:39.788Z INFO  System info {"apifyVersion":"3.5.3","apifyClientVersion":"2.19.0","crawleeVersion":"3.16.0","osType":"Linux","nodeVersion":"v24.15.0"}
[apify.crawler-google-places runId:spwHOiXRDgimae07c] -> 2026-05-06T14:19:40.603Z INFO  [BG ENQUEUE] Finished enqueueing
[apify.crawler-google-places runId:spwHOiXRDgimae07c] ->

  ✓ Saved 164 reviews → sauwandarek_reviews_raw.csv

[29/29] Danau Emtofe (Papua)
  → Scraping dari Apify... (tunggu beberapa menit)


[apify.crawler-google-places runId:BxjEuqnkETJ4ivZkN] -> 2026-05-06T14:20:45.456Z ACTOR: Pulling container image of build t94zr25jkhfIVPY08 from registry.
[apify.crawler-google-places runId:BxjEuqnkETJ4ivZkN] -> 2026-05-06T14:20:45.458Z ACTOR: Creating container.
[apify.crawler-google-places runId:BxjEuqnkETJ4ivZkN] -> 2026-05-06T14:20:45.613Z ACTOR: Starting container.
[apify.crawler-google-places runId:BxjEuqnkETJ4ivZkN] -> 2026-05-06T14:20:45.614Z ACTOR: Running under "LIMITED_PERMISSIONS" permission level.
[apify.crawler-google-places runId:BxjEuqnkETJ4ivZkN] -> 2026-05-06T14:20:46.578Z INFO  System info {"apifyVersion":"3.5.3","apifyClientVersion":"2.19.0","crawleeVersion":"3.16.0","osType":"Linux","nodeVersion":"v24.15.0"}
[apify.crawler-google-places runId:BxjEuqnkETJ4ivZkN] -> 2026-05-06T14:20:47.072Z INFO  [BG ENQUEUE] Finished enqueueing
[apify.crawler-google-places runId:BxjEuqnkETJ4ivZkN] -> 2026-05-06T14:20:47.089Z INFO  HttpCrawler: Starting the crawler.
[apify.crawler-go

  ✓ Saved 119 reviews → danau_emtofe_reviews_raw.csv

BATCH SELESAI


## 4. Ringkasan Hasil

In [5]:
# Ringkasan hasil scraping
df_summary = pd.DataFrame(results_summary)
print(df_summary.to_string(index=False))

ok_count    = len(df_summary[df_summary["status"] == "ok"])
total_new   = df_summary[df_summary["status"] == "ok"]["count"].sum()
skip_exists = len(df_summary[df_summary["status"].str.startswith("skip (exists)")])
skip_nourl  = len(df_summary[df_summary["status"] == "skip (no url)"])
errors      = len(df_summary[df_summary["status"].str.startswith("error")])

print(f"\n{'='*50}")
print(f"Berhasil scrape baru : {ok_count} desa ({total_new} reviews)")
print(f"Skip (sudah ada)     : {skip_exists} desa")
print(f"Skip (no URL)        : {skip_nourl} desa")
print(f"Error                : {errors} desa")
print(f"{'='*50}")
print("\nNEXT STEP:")
print("1. Jalankan Clean_Reviews.ipynb untuk cleaning semua file baru")
print("2. Jalankan SA_AutoLabel.ipynb untuk labeling + aspect detection")
print("3. Dashboard otomatis menampilkan semua desa baru")


                       nama                  provinsi status  count
               Gampong Nusa                      Aceh     ok     61
                      Tomok            Sumatera Utara     ok     31
           Nagari Pariangan            Sumatera Barat     ok    151
                 Buluh Cina                      Riau     ok     97
              Kampung Terih            Kepulauan Riau     ok    151
               Tanjung Laut                     Jambi     ok    119
         Pulau Kumayan Kota                  Bengkulu     ok    144
               Pulau Kemaro          Sumatera Selatan     ok    204
             Mangrove Kurau Kepulauan Bangka Belitung     ok    155
                     Kunjir                   Lampung     ok    147
               Setu Babakan               DKI Jakarta     ok    147
               Kampung Naga                Jawa Barat     ok    137
      Kampung Marengo Baduy                    Banten     ok    147
                  Desa Sade       Nusa Tenggara 

## Troubleshooting

### API Token Error
- Pastikan sudah daftar di https://apify.com
- Buka Settings → Integrations → copy API Token
- Free tier: $5 credit/bulan

### Tidak ada review / 0 results
- Cek URL Google Maps — pastikan URL valid dan tempatnya punya review
- Coba naikkan `maxReviews`
- Cek sisa credit di Apify dashboard

### Review sedikit setelah filter bahasa
- Kurangi `min_match` di `is_indonesian()` (default=2, coba 1)
- Hapus filter `language: "id"` di `run_input` kalau mau ambil semua bahasa dulu, baru filter manual

### Actor error / timeout
- Cek Apify dashboard → Runs → lihat log error
- Pastikan URL Google Maps bukan URL shortlink (pakai URL lengkap)